In [59]:
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.output_parsers import StrOutputParser

llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

llm_model = ChatHuggingFace(llm=llm)

In [60]:
text = "Hello, My name is Somit, "
"Email is somit17@gmail.com and my age is 29"
res = llm_model.invoke(f"Please give me only name, email and age from this text: {text}")
print(type(res.content))

<class 'str'>


# Structured OP

In [61]:
from pydantic import BaseModel,Field
from typing import List

class ResponseStructure(BaseModel):
    name:str = Field(description = "Name")
    email:str= Field(description = "Email Address")
    age:int= Field(description = "Age")

structured_llm  = llm_model.with_structured_output(ResponseStructure,method="json_mode",)

In [62]:

res = structured_llm.invoke(f"Please give me only name, email and age from this text: {text}")

In [63]:
res

{'Name': 'Somit', 'Email': 'Not provided', 'Age': 'Not provided'}

In [64]:
class Movies(BaseModel):
    title:str = Field(description="Movie Title")
    year:int = Field(description="Movie Released Year.")


class AllMovies(BaseModel):
    movies:List[Movies]

movie_llm = llm_model.with_structured_output(Movies,method="json_mode")

In [65]:
res = movie_llm.invoke("Give me Sci-Fi movie")

res

{'Blade Runner 2049': "In a dystopian future, a new blade runner, LAPD Officer K, unearths a long-buried secret that has the potential to plunge what's left of society into chaos."}

In [66]:
res2=llm_model.invoke("Give me Sci-Fi movie")
res2

AIMessage(content="Here's a sci-fi movie recommendation:\n\n**Movie:** Interstellar (2014)\n\n**Directed by:** Christopher Nolan\n\n**Genre:** Sci-Fi, Adventure, Drama\n\n**Plot:**\n\nWhen a wormhole is discovered, a team of astronauts, led by Cooper (Matthew McConaughey), embark on a journey through the wormhole in search of a new habitable planet for humanity. As they travel through space and time, they encounter strange phenomena and confront the consequences of time dilation.\n\n**Why you'll love it:**\n\n* Mind-bending plot that explores the complexities of time and space\n* Stunning visuals and special effects that will leave you breathless\n* Emotional storytelling that explores the themes of love, sacrifice, and the human condition\n\n**Other notable sci-fi movies:**\n\n* Blade Runner (1982)\n* The Matrix (1999)\n* Inception (2010)\n* Arrival (2016)\n* Star Wars: Episode IV - A New Hope (1977)\n\nLet me know if you'd like more recommendations or have specific preferences (e.g.,